# Phase 2.11 Evaluation: TelSano Customer Service Copilot

Runs the full 5-state pipeline against `eval/golden_set.csv` and computes
the four locked metrics from `docs/EVAL.md`:

| Metric | Target |
|---|---|
| Intent classification accuracy | >= 90% |
| Tool selection correctness | >= 85% average |
| Grounding faithfulness (RAGAS) | >= 0.90 average |
| Escalation precision / recall | >= 85% / >= 80% |

**Evaluation strategy**: each row is a fresh single-turn session.
States are driven individually (not via `process_turn`) so intermediate
outputs (classified intent, tools called, escalation decision) can be
captured for scoring.

**Tool selection note**: `expected_tools` in the golden set lists ActState
tools only (`get_billing_info`, `get_customer_account`, `check_network_outage`,
`run_speed_diagnostic`). `create_escalation_ticket` is an EscalateState call
and is evaluated separately via the escalation precision/recall metric.
The scoring function filters both sides to ActState tools before comparing.

In [ ]:
# Install RAGAS for grounding faithfulness scoring.
# RAGAS requires Python 3.10+ (pre-built wheels). This cell is a no-op
# if RAGAS is already installed.
# Grounding scoring also requires an LLM API key -- see Cell 13.
!pip install -q ragas>=0.2.0

In [ ]:
import sys
import asyncio
import json
import csv
import time
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import pandas as pd

# Ensure project root is on sys.path so src.* imports resolve.
# Adjust this path if the notebook is run from a different working directory.
PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import get_config
from src.orchestrator.agents.factory import AgentFactory
from src.orchestrator.models import SessionState, StateContext, RoutingDecision
from src.orchestrator.state_machine import StateMachine, _ACT_DECISIONS

print(f"Python {sys.version}")
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# -----------------------------------------------------------------------
# CONFIGURATION
# Set LIMIT to an integer to evaluate a subset (e.g. 10 for a quick run).
# Set LIMIT to None to evaluate all 100 queries.
# -----------------------------------------------------------------------
LIMIT = 10

GOLDEN_SET_PATH = PROJECT_ROOT / "eval" / "golden_set.csv"
OUTPUT_DIR = PROJECT_ROOT / "eval"
RUN_DATE = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M")
OUTPUT_PATH = OUTPUT_DIR / f"results_{RUN_DATE}.csv"

print(f"Golden set: {GOLDEN_SET_PATH}")
print(f"Output:     {OUTPUT_PATH}")
print(f"LIMIT:      {LIMIT}")

In [ ]:
with open(GOLDEN_SET_PATH, encoding="utf-8", newline="") as f:
    all_rows = list(csv.DictReader(f))

rows_to_run = all_rows[:LIMIT] if LIMIT is not None else all_rows

print(f"Total queries in golden set: {len(all_rows)}")
print(f"Queries to run:              {len(rows_to_run)}")

# Show composition of rows being evaluated
from collections import Counter
intent_counts = Counter(r["expected_intent"] for r in rows_to_run)
cat_counts = Counter(r["category"] for r in rows_to_run)
print(f"\nComposition: {dict(cat_counts)}")
print(f"Intents:     {dict(intent_counts)}")

## Authentication

`AgentFactory` uses `DeviceCodeCredential`. When the next cell runs,
a URL and one-time code will be printed. Open the URL in a browser,
enter the code, and sign in with your Azure account. The credential
is cached for the lifetime of the session.

The code typically expires after 60 seconds -- authenticate promptly.

In [ ]:
config = get_config()
factory = AgentFactory(config)
machine = StateMachine(factory)

print("StateMachine initialized.")
print(f"Endpoint: {config.AZURE_FOUNDRY_PROJECT_ENDPOINT[:60]}...")
print(f"Vector store: {config.VECTOR_STORE_ID}")

In [ ]:
# ActState tool names: the only tools tracked in ActOutput.tools_called.
# create_escalation_ticket is called by EscalateState and is evaluated
# separately via the escalation precision/recall metric.
_ACT_TOOL_NAMES = frozenset({
    "get_billing_info",
    "get_customer_account",
    "check_network_outage",
    "run_speed_diagnostic",
})


def score_tools(actual: list[str], expected: list[str]) -> float:
    """Score tool selection per EVAL.md rules.

    Both lists are pre-filtered to ActState tools only. Order matters
    for the 1.0 score (correct sequence required for multi-tool cases).

    Returns:
        1.0  exact match including order
        0.5  all required tools present but wrong order or extra tools called
        0.0  a required tool is missing, wrong tool called, or tools called
             when none were expected
    """
    if actual == expected:
        return 1.0
    if not expected:
        # Tools were called when none were expected
        return 0.0
    if not all(t in actual for t in expected):
        # A required tool is absent or wrong tool was called instead
        return 0.0
    # All required tools present but wrong order or extra tools
    return 0.5


# Smoke-test the scorer
assert score_tools([], []) == 1.0
assert score_tools(["get_billing_info"], ["get_billing_info"]) == 1.0
assert score_tools(["check_network_outage", "run_speed_diagnostic"],
                   ["check_network_outage", "run_speed_diagnostic"]) == 1.0
assert score_tools(["run_speed_diagnostic", "check_network_outage"],
                   ["check_network_outage", "run_speed_diagnostic"]) == 0.5
assert score_tools(["check_network_outage", "run_speed_diagnostic", "get_billing_info"],
                   ["check_network_outage", "run_speed_diagnostic"]) == 0.5
assert score_tools([], ["get_billing_info"]) == 0.0
assert score_tools(["get_billing_info"], []) == 0.0
assert score_tools(["get_customer_account"], ["get_billing_info"]) == 0.0
print("score_tools: all assertions passed.")

In [ ]:
async def evaluate_row(row: dict) -> dict:
    """Run one golden-set row through the pipeline and return a scored result dict."""
    query = row["query"]
    now = datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")
    account_id = row["customer_account_id"] or None

    session = SessionState(
        session_id=str(uuid4()),
        correlation_id=str(uuid4()),
        account_id=account_id,
        conversation_history=[],
        started_at=now,
        last_updated=now,
    )

    t_start = time.monotonic()
    actual_intent = ""
    actual_tools: list[str] = []
    actual_escalation = False
    actual_answer = ""
    actual_citations: list[str] = []
    routing_decision_str = ""
    error = ""

    try:
        context = StateContext(session_state=session, customer_message=query)

        # --- ClassifyState ---
        classify_output = await machine._classify.run(context)
        actual_intent = classify_output.intent
        context = context.model_copy(update={"classify_output": classify_output})

        # --- RouteState ---
        routing_decision = await machine._route.run(context)
        routing_decision_str = routing_decision.value
        context = context.model_copy(update={"routing_decision": routing_decision})

        # --- ActState (content paths only) ---
        act_output = None
        act_failed = False
        if routing_decision in _ACT_DECISIONS:
            try:
                act_output = await machine._act.run(context)
                context = context.model_copy(update={"act_output": act_output})
                actual_tools = [r.tool_name for r in act_output.tools_called]
            except Exception as act_exc:
                act_failed = True
                error = f"ActState: {act_exc}"

        # --- EscalateState ---
        run_escalate = (
            routing_decision == RoutingDecision.SKIP_TO_ESCALATE
            or act_failed
            or (act_output is not None
                and act_output.resolution_status == "unresolved")
        )
        if run_escalate:
            escalate_output = await machine._escalate.run(context)
            context = context.model_copy(update={"escalate_output": escalate_output})
            actual_escalation = True

        # --- RespondState (always runs) ---
        respond_output = await machine._respond.run(context)
        actual_answer = respond_output.message
        actual_citations = respond_output.citations

    except Exception as exc:
        if not error:
            error = str(exc)

    latency_ms = int((time.monotonic() - t_start) * 1000)

    # --- Scoring ---
    expected_intent = row["expected_intent"]
    expected_escalation = row["expected_escalation"] == "true"

    # Filter both tool lists to ActState tools only before scoring.
    # create_escalation_ticket is an EscalateState call, evaluated via
    # escalation precision/recall instead.
    raw_expected = json.loads(row["expected_tools"])
    expected_act_tools = [t for t in raw_expected if t in _ACT_TOOL_NAMES]
    actual_act_tools = [t for t in actual_tools if t in _ACT_TOOL_NAMES]

    intent_correct = 1 if actual_intent == expected_intent else 0
    tool_score = score_tools(actual_act_tools, expected_act_tools)
    escalation_correct = actual_escalation == expected_escalation

    return {
        # Pass through all golden set columns
        **row,
        # Actual pipeline outputs
        "actual_intent": actual_intent,
        "actual_tools": json.dumps(actual_tools),
        "actual_escalation": str(actual_escalation).lower(),
        "routing_decision": routing_decision_str,
        "actual_answer": actual_answer,
        "actual_citations": json.dumps(actual_citations),
        # Per-row scores
        "intent_correct": intent_correct,
        "tool_score": tool_score,
        "grounding_score": "",  # filled in by the RAGAS cell below
        "escalation_correct": str(escalation_correct).lower(),
        "latency_ms": latency_ms,
        "error": error,
    }

print("evaluate_row defined.")

In [ ]:
results: list[dict] = []

async def run_evaluation() -> None:
    total = len(rows_to_run)
    for i, row in enumerate(rows_to_run, 1):
        result = await evaluate_row(row)
        results.append(result)
        status = "OK" if not result["error"] else f"ERR: {result['error'][:50]}"
        print(
            f"[{i:3d}/{total}] {row['query_id']:8s} "
            f"intent={result['actual_intent']:10s} "
            f"tools={result['actual_tools']:30s} "
            f"esc={result['actual_escalation']:5s} "
            f"{result['latency_ms']:5d}ms  {status}"
        )

await run_evaluation()
print(f"\nEvaluation complete. {len(results)} rows processed.")

In [ ]:
df = pd.DataFrame(results)
df["tool_score"] = df["tool_score"].astype(float)
df["intent_correct"] = df["intent_correct"].astype(int)
df["latency_ms"] = df["latency_ms"].astype(int)

print(f"DataFrame shape: {df.shape}")
df[[
    "query_id", "expected_intent", "actual_intent",
    "expected_tools", "actual_tools",
    "expected_escalation", "actual_escalation",
    "intent_correct", "tool_score", "latency_ms", "error"
]].head(10)

In [ ]:
# --- Metric 1: Intent classification accuracy ---
intent_accuracy = df["intent_correct"].mean() * 100
passed_intent = "PASS" if intent_accuracy >= 90 else "FAIL"

# --- Metric 2: Tool selection correctness ---
# Exclude escalate-intent rows: their tool_score is always 1.0 after
# filtering create_escalation_ticket, which would inflate the metric.
tool_df = df[df["expected_intent"] != "escalate"]
tool_avg = tool_df["tool_score"].mean() * 100 if len(tool_df) > 0 else float("nan")
passed_tools = "PASS" if tool_avg >= 85 else "FAIL"

# --- Metric 3: Grounding faithfulness (RAGAS) ---
# Populated by the RAGAS cell below; shown as PENDING until that cell runs.
grounding_rows = df[df["actual_citations"] != "[]"]
gs = pd.to_numeric(grounding_rows["grounding_score"], errors="coerce")
grounding_avg = gs.mean() if gs.notna().any() else float("nan")
if pd.isna(grounding_avg):
    passed_grounding = "PENDING (run RAGAS cell)"
else:
    passed_grounding = "PASS" if grounding_avg >= 0.90 else "FAIL"

# --- Metric 4: Escalation precision and recall ---
expected_esc = df["expected_escalation"] == "true"
actual_esc = df["actual_escalation"] == "true"
tp = int((expected_esc & actual_esc).sum())
fp = int((~expected_esc & actual_esc).sum())
fn = int((expected_esc & ~actual_esc).sum())
precision = tp / (tp + fp) * 100 if (tp + fp) > 0 else float("nan")
recall    = tp / (tp + fn) * 100 if (tp + fn) > 0 else float("nan")
passed_precision = "PASS" if precision >= 85 else "FAIL"
passed_recall    = "PASS" if recall    >= 80 else "FAIL"

# --- Latency p95 ---
p95_latency = df["latency_ms"].quantile(0.95)
passed_latency = "PASS" if p95_latency <= 5000 else "FAIL"

# --- Deflection rate on standard set ---
std_df = df[df["category"] == "standard"]
deflection = (std_df["actual_escalation"] != "true").mean() * 100 if len(std_df) > 0 else float("nan")

print("=" * 65)
print("AGGREGATE RESULTS")
print("=" * 65)
print(f"Intent accuracy         {intent_accuracy:6.1f}%   target >=90%    {passed_intent}")
print(f"Tool selection avg      {tool_avg:6.1f}%   target >=85%    {passed_tools}")
print(f"Grounding (RAGAS)       {grounding_avg:6.3f}    target >=0.90   {passed_grounding}")
print(f"Escalation precision    {precision:6.1f}%   target >=85%    {passed_precision}")
print(f"Escalation recall       {recall:6.1f}%   target >=80%    {passed_recall}")
print(f"Latency p95             {p95_latency:6.0f}ms  target <=5000ms {passed_latency}")
print(f"Deflection rate (std)   {deflection:6.1f}%   target 30-40%")
print(f"Errors                  {(df['error'] != '').sum()} rows")
print("=" * 65)
print(f"Escalation TP={tp}  FP={fp}  FN={fn}")

In [ ]:
# RAGAS grounding faithfulness scoring.
#
# Requires an LLM API key. RAGAS defaults to OpenAI; to use Azure OpenAI,
# configure ragas.llms before calling evaluate().
# Set OPENAI_API_KEY in the Colab environment or secrets panel before running.
#
# Skip this cell if running without LLM configuration.

import os

try:
    from datasets import Dataset
    from ragas import evaluate as ragas_evaluate
    from ragas.metrics import faithfulness

    grounding_data = [
        {
            "question": r["query"],
            "answer": r["actual_answer"],
            "contexts": json.loads(r["actual_citations"]),
        }
        for _, r in df.iterrows()
        if json.loads(r["actual_citations"])
    ]

    if grounding_data:
        ragas_ds = Dataset.from_dict({
            "question": [d["question"] for d in grounding_data],
            "answer":   [d["answer"]   for d in grounding_data],
            "contexts": [d["contexts"] for d in grounding_data],
        })
        ragas_result = ragas_evaluate(ragas_ds, metrics=[faithfulness])
        ragas_scores = ragas_result["faithfulness"]

        # Write scores back to df for output CSV and metric re-computation
        grounding_idx = [
            i for i, r in df.iterrows()
            if json.loads(r["actual_citations"])
        ]
        for idx, score in zip(grounding_idx, ragas_scores):
            df.at[idx, "grounding_score"] = score

        mean_score = sum(ragas_scores) / len(ragas_scores)
        passed = "PASS" if mean_score >= 0.90 else "FAIL"
        print(f"Grounding faithfulness: {mean_score:.3f}  ({len(ragas_scores)} rows scored)  {passed}")
    else:
        print("No KB-grounded rows in this run; RAGAS scoring skipped.")

except ImportError:
    print("RAGAS not available. Run: pip install ragas>=0.2.0")
except Exception as ragas_exc:
    print(f"RAGAS scoring failed: {ragas_exc}")
    print("Check OPENAI_API_KEY is set and the RAGAS API is compatible with the installed version.")

In [ ]:
# -----------------------------------------------------------------------
# FAILURE ANALYSIS
# Surfaces the top error categories for prioritized iteration.
# -----------------------------------------------------------------------

print("INTENT CLASSIFICATION FAILURES")
intent_fail = df[df["intent_correct"] == 0][
    ["query_id", "query", "expected_intent", "actual_intent",
     "category", "adversarial_type"]
].reset_index(drop=True)
print(f"{len(intent_fail)} failures")
display(intent_fail) if len(intent_fail) > 0 else print("None.")

print("\nTOOL SELECTION FAILURES (score < 1.0)")
tool_fail = df[(df["tool_score"] < 1.0) & (df["expected_intent"] != "escalate")][
    ["query_id", "query", "expected_tools", "actual_tools", "tool_score"]
].reset_index(drop=True)
print(f"{len(tool_fail)} rows with imperfect tool score")
display(tool_fail) if len(tool_fail) > 0 else print("None.")

print("\nESCALATION MISMATCHES")
esc_fail = df[df["escalation_correct"] == "false"][
    ["query_id", "query", "expected_escalation", "actual_escalation",
     "category", "adversarial_type"]
].reset_index(drop=True)
print(f"{len(esc_fail)} escalation mismatches")
display(esc_fail) if len(esc_fail) > 0 else print("None.")

print("\nROWS WITH PIPELINE ERRORS")
err_rows = df[df["error"] != ""][["query_id", "query", "error"]].reset_index(drop=True)
display(err_rows) if len(err_rows) > 0 else print("None.")

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)
print(f"Results written to: {OUTPUT_PATH}")
print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")